<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Supervised_Learning_lektioner/Lab_3_Lektion_4_Confusion_Matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Supervised Learning – Lektion 4: Confusion Matrix och Evaluering

**Målgrupp:** Gymnasiet, 16 år  
**Tid:** ca 60 minuter  
**Mål:** Förstå Confusion Matrix, Precision, Recall och F1-score. Se VAR modellen gör fel.

---

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/  
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, 
                               precision_score, recall_score, f1_score,
                               classification_report)

print("✅ Alla bibliotek importerade!")

# Ladda och förbered data
iris = load_iris()
X, y = iris.data, iris.target
class_names = ['Setosa', 'Versicolor', 'Virginica']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Träna en standardmodell
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("✅ Modell tränad och redo!")
print(f"   Testset: {len(y_test)} blommor")
print(f"   Accuracy: {accuracy_score(y_test, y_pred):.1%}")

---
## ⚠️ Del 1 – Accuracy räcker inte alltid!

I lektion 3 använde vi **accuracy** för att mäta modellens prestanda.

```
Accuracy = Antal rätt / Totalt antal = 90% ✓
```

Men vad händer om:
- 95% av alla e-post är legitim (inte spam)
- En modell säger "ALLT ÄR LEGITIM" på alla mail

**Accuracy = 95%! 🎉**

Men modellen hittar **INGET SPAM** – den är värdelös! 😱

Vi behöver bättre mått. Det är här **Confusion Matrix** kommer in!

---

### 🧩 Quiz 4.1

1. Om 99% av kreditkort-transaktioner är legitima och modellen säger "allt legitimt" – vad är accuracy?
2. Är en sådan modell bra? Varför/varför inte?

---
## 📊 Del 2 – Confusion Matrix: Vad är det?

En **Confusion Matrix** visar EXAKT hur modellen förutsäger varje klass:

### För ett 2-klassificeringsproblem (t.ex. Spam/Inte spam):

```
                    FÖRUTSATT
                  Spam    Inte spam
FAKTISK  Spam   [ TP  |   FN  ]
         Inte  [ FP  |   TN  ]
```

| Term | Fullständigt namn | Förklaring |
|------|------------------|-----------|
| **TP** | True Positive | Spam – modellen sa Spam ✓ |
| **TN** | True Negative | Inte spam – modellen sa Inte spam ✓ |
| **FP** | False Positive | Inte spam – modellen sa Spam ✗ (Falskt alarm) |
| **FN** | False Negative | Spam – modellen sa Inte spam ✗ (Missat spam) |

### För Iris (3 klasser):

```
                   FÖRUTSATT
              Setosa  Versicolor  Virginica
FAKTISK  Setosa [  10  |     0    |    0   ]
         Versi  [   0  |     9    |    1   ]  ← Fel! Virginica förutsades som Versicolor
         Virgi  [   0  |     0    |   10   ]
```

Diagonalen (↖↘) = rätt svar!  
Utanför diagonalen = fel svar!

In [ ]:
# ============================================================
# INTERAKTIV CONFUSION MATRIX HEATMAP
# ============================================================

output_cm = widgets.Output()

def rita_confusion_matrix(max_depth):
    with output_cm:
        clear_output(wait=True)
        
        # Träna modell med vald max_depth
        m = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
        m.fit(X_train, y_train)
        pred = m.predict(X_test)
        
        cm = confusion_matrix(y_test, pred)
        acc = accuracy_score(y_test, pred)
        
        # Rita heatmap
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        
        # ── Confusion matrix ────────────────────────────────
        ax = axes[0]
        im = ax.imshow(cm, cmap='Blues', aspect='auto')
        
        ax.set_xticks(range(3))
        ax.set_yticks(range(3))
        ax.set_xticklabels(class_names, fontsize=11)
        ax.set_yticklabels(class_names, fontsize=11)
        ax.set_xlabel('Förutsagd art', fontsize=12, fontweight='bold')
        ax.set_ylabel('Faktisk art', fontsize=12, fontweight='bold')
        ax.set_title(f'Confusion Matrix (max_depth={max_depth})', fontsize=13, fontweight='bold')
        
        # Siffror i cellerna
        for i in range(3):
            for j in range(3):
                val = cm[i, j]
                color = 'white' if val > cm.max() * 0.5 else 'black'
                ax.text(j, i, str(val), ha='center', va='center', 
                       fontsize=16, fontweight='bold', color=color)
        
        plt.colorbar(im, ax=ax, shrink=0.8)
        
        # ── Metrics per klass ────────────────────────────────
        ax2 = axes[1]
        precision = precision_score(y_test, pred, average=None, zero_division=0)
        recall = recall_score(y_test, pred, average=None, zero_division=0)
        f1 = f1_score(y_test, pred, average=None, zero_division=0)
        
        x = np.arange(3)
        width = 0.25
        
        ax2.bar(x - width, precision * 100, width, label='Precision', color='#4CAF50', alpha=0.8)
        ax2.bar(x, recall * 100, width, label='Recall', color='#2196F3', alpha=0.8)
        ax2.bar(x + width, f1 * 100, width, label='F1-score', color='#FF9800', alpha=0.8)
        
        ax2.set_xticks(x)
        ax2.set_xticklabels(class_names, fontsize=11)
        ax2.set_ylabel('Värde (%)', fontsize=11)
        ax2.set_title(f'Precision / Recall / F1 per klass\n(Accuracy: {acc:.1%})', 
                      fontsize=12, fontweight='bold')
        ax2.legend(fontsize=10)
        ax2.set_ylim(0, 110)
        ax2.grid(True, alpha=0.3, axis='y')
        
        # Siffror på staplarna
        for bars in [ax2.containers[0], ax2.containers[1], ax2.containers[2]]:
            for bar in bars:
                h = bar.get_height()
                if h > 0:
                    ax2.text(bar.get_x() + bar.get_width()/2, h + 1, 
                            f'{h:.0f}%', ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout()
        plt.show()
        
        # Textsummering
        print(f"📊 Sammanfattning (max_depth={max_depth}):")
        print(f"   Accuracy: {acc:.1%}")
        print()
        print(f"{'Art':>12} | {'Precision':>10} | {'Recall':>10} | {'F1-score':>10}")
        print("-" * 50)
        for i, cn in enumerate(class_names):
            print(f"{cn:>12} | {precision[i]:>9.1%} | {recall[i]:>9.1%} | {f1[i]:>9.1%}")
        
        # Hitta eventuella fel
        errors = []
        for i in range(3):
            for j in range(3):
                if i != j and cm[i, j] > 0:
                    errors.append(f"   {cm[i,j]} {class_names[i]}-blommor förutsades som {class_names[j]}")
        if errors:
            print()
            print("❌ Fel modellen gör:")
            for e in errors:
                print(e)
        else:
            print()
            print("✅ Inga fel! Perfekt klassificering!")

depth_slider_cm = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='max_depth:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

widgets.interactive(rita_confusion_matrix, max_depth=depth_slider_cm)

---
## 📏 Del 3 – Precision, Recall och F1-score

### Precision

> *"Av de blommor jag SA var Setosa, hur många VAR faktiskt Setosa?"*

```
Precision = TP / (TP + FP)

Exempel: Modellen sa "Setosa" 11 gånger.
         10 var rätt (TP), 1 var fel (FP)
         Precision = 10 / (10+1) = 90.9%
```

### Recall (Sensitivity)

> *"Av alla FAKTISKA Setosa-blommor, hur många HITTADE jag?"*

```
Recall = TP / (TP + FN)

Exempel: Det finns 10 Setosa-blommor i testset.
         Modellen hittade 9 (TP=9), missade 1 (FN=1)
         Recall = 9 / (9+1) = 90%
```

### F1-score

> *"En balanserad kombination av Precision och Recall"*

```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```

---

### 🏥 Viktigt exempel – Medicinsk diagnos:

| Situation | Vad är viktigast? |
|-----------|------------------|
| Cancerscreening | **Recall** – Bättre att ha falskt alarm än missa cancer |
| Spam-filter | **Precision** – Bättre att missa spam än blockera viktigt mail |
| Fraud Detection | **Recall** – Bättre att falskt alarma än missa bedrägeri |

---

### 🧩 Quiz 4.2

En modell undersöker 100 patienter för en sjukdom.
- 20 patienter HAR sjukdomen
- Modellen säger "sjuk" för 15 av dessa (TP=15, FN=5)
- Modellen säger "sjuk" felaktigt för 10 friska (FP=10, TN=70)

1. Recall = TP / (TP + FN) = ___ / (___ + ___) = ___
2. Precision = TP / (TP + FP) = ___ / (___ + ___) = ___
3. Är det här en bra medicinsk diagnostik? Varför?

In [ ]:
# ============================================================
# INTERAKTIV PRECISION/RECALL SIMULATOR
# ============================================================

output_pr = widgets.Output()

def simulera_precision_recall(tp, fp, fn, tn):
    with output_pr:
        clear_output(wait=True)
        
        total = tp + fp + fn + tn
        
        if tp + fp == 0 or tp + fn == 0:
            print("⚠️ Ogiltiga värden!")
            return
        
        acc = (tp + tn) / total
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        # Rita confusion matrix 2x2
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Confusion matrix
        cm_2x2 = np.array([[tn, fp], [fn, tp]])
        ax = axes[0]
        im = ax.imshow(cm_2x2, cmap='Blues')
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Förutsatt NEJ', 'Förutsatt JA'], fontsize=11)
        ax.set_yticklabels(['Faktiskt NEJ', 'Faktiskt JA'], fontsize=11)
        ax.set_xlabel('Förutsägelse', fontsize=11)
        ax.set_ylabel('Faktiskt', fontsize=11)
        ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
        
        labels = [[f'TN={tn}', f'FP={fp}'], [f'FN={fn}', f'TP={tp}']]
        colors_cm = [['#C8E6C9', '#FFCDD2'], ['#FFCDD2', '#C8E6C9']]
        for i in range(2):
            for j in range(2):
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1, 
                                           color=colors_cm[i][j], alpha=0.5))
                ax.text(j, i, labels[i][j], ha='center', va='center', 
                       fontsize=13, fontweight='bold')
        
        plt.colorbar(im, ax=ax, shrink=0.8)
        
        # Metrics bars
        ax2 = axes[1]
        metrics = {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1-score': f1}
        met_colors = ['#9C27B0', '#4CAF50', '#2196F3', '#FF9800']
        
        bars = ax2.bar(metrics.keys(), [v*100 for v in metrics.values()], 
                       color=met_colors, alpha=0.8, edgecolor='black', linewidth=0.5)
        ax2.set_ylim(0, 115)
        ax2.set_ylabel('Värde (%)', fontsize=11)
        ax2.set_title('Evalueringsmått', fontsize=13, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        for bar, val in zip(bars, metrics.values()):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        print(f"📊 Beräknade mått:")
        print(f"   Accuracy:  {acc:.1%}  = ({tp}+{tn}) / {total}")
        print(f"   Precision: {precision:.1%}  = {tp} / ({tp}+{fp})")
        print(f"   Recall:    {recall:.1%}  = {tp} / ({tp}+{fn})")
        print(f"   F1-score:  {f1:.1%}  = 2 × ({precision:.2f} × {recall:.2f}) / ({precision:.2f}+{recall:.2f})")

style = {'description_width': 'initial'}
layout = widgets.Layout(width='300px')

tp_sl = widgets.IntSlider(value=8, min=0, max=20, description='TP (Rätt positiva):', style=style, layout=layout)
fp_sl = widgets.IntSlider(value=2, min=0, max=20, description='FP (Falskt larm):', style=style, layout=layout)
fn_sl = widgets.IntSlider(value=3, min=0, max=20, description='FN (Missade):', style=style, layout=layout)
tn_sl = widgets.IntSlider(value=15, min=0, max=50, description='TN (Rätt negativa):', style=style, layout=layout)

widgets.interactive(simulera_precision_recall, tp=tp_sl, fp=fp_sl, fn=fn_sl, tn=tn_sl)

---
### ✏️ Övning 3.1 – Precision vs Recall

Använd simulatorn ovan för att svara:

1. **Spam-filter:** Sätt FP=0 (inga falskt alarm). Vad händer med Recall?
   - TP=8, FP=0, FN=5, TN=17 → Recall = ___, Precision = ___
   
2. **Medicinsk test:** Sätt FN=0 (missa inga sjuka). Vad händer med Precision?
   - TP=11, FP=10, FN=0, TN=10 → Recall = ___, Precision = ___

3. Förklara trade-off:en mellan Precision och Recall med egna ord.

---
## 🌸 Del 4 – Analysera Iris-modellen

Nu tittar vi på vår tränade modell och analyserar vilka arter som är svårast att klassificera.

In [ ]:
# ============================================================
# FULLSTÄNDIG ANALYS AV IRIS-MODELLEN
# ============================================================

output_full = widgets.Output()

def fullanalys(max_depth):
    with output_full:
        clear_output(wait=True)
        
        m = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
        m.fit(X_train, y_train)
        pred = m.predict(X_test)
        
        cm = confusion_matrix(y_test, pred)
        
        print(f"🌳 Beslutsträd med max_depth={max_depth}")
        print()
        print(classification_report(y_test, pred, target_names=class_names))
        
        # Hitta felklassificerade blommor
        print("❌ Felklassificerade blommor:")
        n_errors = 0
        for i, (true, predicted) in enumerate(zip(y_test, pred)):
            if true != predicted:
                n_errors += 1
                feats = X_test[i]
                print(f"   Blomma {i+1}: Faktisk={class_names[true]}, Förutsagd={class_names[predicted]}")
                print(f"            [sepal: {feats[0]:.1f}×{feats[1]:.1f}, petal: {feats[2]:.1f}×{feats[3]:.1f}]")
        
        if n_errors == 0:
            print("   Inga fel! ✅")
        else:
            print(f"   Totalt {n_errors} fel av {len(y_test)} blommor")

depth_sl_full = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='max_depth:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

widgets.interactive(fullanalys, max_depth=depth_sl_full)

---
### ✏️ Övning 4.1 – Analysera iris-modellen

Kör fullanalysen ovan med max_depth=3 och svara:

1. Vilken art är SVÅRAST att klassificera? → ___
2. Vilka arter blandas oftast ihop? → ___
3. Vad är precision för Setosa? → ___
4. Vad är recall för Virginica? → ___
5. Varför är Setosa lättast att klassificera? (Tips: kom ihåg lektion 2!) → ___

---
## 🔬 Del 5 – Experimentera!

Prova olika inställningar och se hur Precision/Recall ändras.

In [ ]:
# ============================================================
# EXPERIMENTERA MED HYPERPARAMETRAR
# ============================================================

output_exp = widgets.Output()

def experimentera(max_depth, criterion, min_samples_leaf):
    with output_exp:
        clear_output(wait=True)
        
        m = DecisionTreeClassifier(
            max_depth=max_depth, 
            criterion=criterion,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        m.fit(X_train, y_train)
        pred = m.predict(X_test)
        
        acc = accuracy_score(y_test, pred)
        prec = precision_score(y_test, pred, average='macro', zero_division=0)
        rec = recall_score(y_test, pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, pred, average='macro', zero_division=0)
        
        cm = confusion_matrix(y_test, pred)
        
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names,
                    linewidths=0.5, ax=ax, annot_kws={'size': 13, 'weight': 'bold'})
        ax.set_xlabel('Förutsagd', fontsize=11)
        ax.set_ylabel('Faktisk', fontsize=11)
        ax.set_title(f'Confusion Matrix\n(depth={max_depth}, criterion={criterion}, min_leaf={min_samples_leaf})',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print(f"📊 Mått (genomsnitt för alla klasser):")
        print(f"   Accuracy:  {acc:.1%}")
        print(f"   Precision: {prec:.1%}")
        print(f"   Recall:    {rec:.1%}")
        print(f"   F1-score:  {f1:.1%}")

style = {'description_width': 'initial'}
depth_sl2 = widgets.IntSlider(value=3, min=1, max=10, step=1, 
    description='max_depth:', style=style, layout=widgets.Layout(width='400px'))
criterion_dd = widgets.Dropdown(options=['gini', 'entropy'], value='gini',
    description='Criterion:', style=style, layout=widgets.Layout(width='300px'))
min_leaf_sl = widgets.IntSlider(value=1, min=1, max=20, step=1,
    description='min_samples_leaf:', style=style, layout=widgets.Layout(width='400px'))

widgets.interactive(experimentera, max_depth=depth_sl2, criterion=criterion_dd, 
                    min_samples_leaf=min_leaf_sl)

---
## 🧩 Quiz – Lektion 4

**Fråga 1:** Vad är en "True Positive"? Ge ett exempel med Iris-datasetet.

*Ditt svar:* ___

---

**Fråga 2:** En modell har Precision=100% men Recall=50% för klass "Virginica". Vad betyder det?

*Ditt svar:* ___

---

**Fråga 3:** Titta på confusion matrix. Om raden för "Versicolor" är [0, 8, 2], hur många Versicolor-blommor klassificerades som Virginica?

*Ditt svar:* ___

---

**Fråga 4:** Varför är diagonalen i confusion matrix viktig?

*Ditt svar:* ___

---

**Fråga 5:** I ett cancerscreeningssystem: Vad är ett "False Negative" och varför är det farligt?

*Ditt svar:* ___

---

**Fråga 6:** Vad är F1-score och när används det?

*Ditt svar:* ___

---

**Fråga 7:** En modell har accuracy=95%, men recall för klassen "bedrägeri"=10%. Är det en bra modell? Varför?

*Ditt svar:* ___

---

**Fråga 8:** Vad händer med confusion matrix om vi minskar max_depth till 1? (Prova och beskriv!)

*Ditt svar:* ___

---

**Fråga 9:** Vad menas med "macro average" i classification_report?

*Ditt svar:* ___

---

**Fråga 10:** Vad är skillnaden mellan precision och recall med egna ord?

*Ditt svar:* ___

---

**Fråga 11:** Varför blandas ofta Versicolor och Virginica ihop men inte Setosa?

*Ditt svar:* ___

---

**Fråga 12:** Om precision för en klass är 0%, vad säger det om modellens beteende?

*Ditt svar:* ___

---

### ✅ Sammanfattning – Vad du lärt dig:

- 📊 **Confusion Matrix** = Visar exakt var modellen gör fel
- 🎯 **Precision** = Av alla jag sa var X, hur många var faktiskt X?
- 🔍 **Recall** = Av alla X, hur många hittade jag?
- ⚖️ **F1-score** = Balanserat mått mellan precision och recall
- ⚠️ **Trade-off** = Öka recall → minska precision (och vice versa)

**Nästa lektion:** Fraud Detection – ett verkligt ML-problem! 💳